# Oakland Businesses → Business Improvement District (BID) Spatial Join

This notebook:
1. Reads `../../data/oakland_businesses_combined.csv` and converts it into a point `GeoDataFrame` using the `latitude`/`longitude` fields.
2. Reads `../../data/geo/block_groups/OAK_BIDs.geojson` as a polygon `GeoDataFrame`.
3. Performs a spatial join, keeping only business points that fall within a BID polygon.
4. Attaches the `FID` and `BID` fields from the polygon layer to the resulting point `GeoDataFrame`.
5. For a set of categorical fields (`broad_sector`, `employee_range`, `sales_range`, `own_or_lease_da`), computes the percentage makeup of each category within each BID — excluding null values for that field — and saves one standalone CSV per field, keyed by `FID`/`BID`, so each can be joined back to the polygon file later.

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

## 1. Load businesses CSV as a point GeoDataFrame

In [ ]:
businesses_path = "../../data/business/combined/oakland_businesses_combined.csv"

df = pd.read_csv(businesses_path)

# Drop rows missing coordinates, since they can't be turned into points
df = df.dropna(subset=["latitude", "longitude"]).copy()

geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]

businesses_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print(f"Loaded {len(businesses_gdf)} business points")
businesses_gdf.head()

## 2. Load BID polygons

In [ ]:
bids_path = "../../data/geo/block_groups/OAK_BIDs.geojson"

bids_gdf = gpd.read_file(bids_path)

print(f"Loaded {len(bids_gdf)} BID polygons")
print("Columns:", list(bids_gdf.columns))
bids_gdf.head()

## 3. Align CRS

Make sure both layers share the same coordinate reference system before joining.

In [ ]:
if businesses_gdf.crs != bids_gdf.crs:
    businesses_gdf = businesses_gdf.to_crs(bids_gdf.crs)

print("Businesses CRS:", businesses_gdf.crs)
print("BIDs CRS:", bids_gdf.crs)

## 4. Spatial join: keep only businesses that fall within a BID polygon

`predicate="within"` keeps only the business points located inside a BID polygon (an inner join, so points outside every polygon are dropped). The `FID` and `BID` fields from the polygon layer are attached to each matching point.

In [ ]:
# Only keep the polygon fields we need for the join (plus geometry), to avoid
# column-name collisions with the businesses table.
ID_COLS = ["FID", "BID"]

bids_join_cols = [c for c in ID_COLS + ["geometry"] if c in bids_gdf.columns]
missing = [c for c in ID_COLS if c not in bids_gdf.columns]
if missing:
    print(f"Warning: expected field(s) not found in BID polygon file: {missing}")
    print("Available columns:", list(bids_gdf.columns))

bids_subset = bids_gdf[bids_join_cols]

businesses_in_bids = gpd.sjoin(
    businesses_gdf,
    bids_subset,
    how="inner",
    predicate="within",
)

# Drop the sjoin index bookkeeping column, if present
businesses_in_bids = businesses_in_bids.drop(columns=["index_right"], errors="ignore")

print(f"{len(businesses_in_bids)} of {len(businesses_gdf)} businesses fall within a BID polygon")
businesses_in_bids.head()

## 5. Inspect result

In [ ]:
businesses_in_bids[ID_COLS].value_counts()

## 6. Per-BID category distributions

One function, reused for every categorical field of interest: `broad_sector`, `employee_range`, `sales_range`, and `own_or_lease_da`.

For a given field, this:
1. Drops rows where that field is null (percentages are computed over non-null businesses only, not all businesses in the BID).
2. Counts businesses per `FID`/`BID` x category.
3. Converts counts to percentages of the (non-null) total within each `FID`/`BID`.
4. Pivots to one row per `FID`/`BID`, one column per category (named after the category's raw value).

In [ ]:
import re


def _normalize_label(text):
    """Loose normalization used only to MATCH labels to a canonical form.

    Collapses runs of whitespace to a single space, trims ends, and forces a
    single space on each side of any hyphen, so e.g. "$100-500 Million",
    "$100 -500  Million", and "$100 - 500 Million" all match the same key.
    This is for lookup purposes only -- the canonical, standardized label
    (from `category_order`) is what actually gets written to the output.
    """
    text = re.sub(r"\s+", " ", str(text).strip())
    text = re.sub(r"\s*-\s*", " - ", text)
    return text


def build_category_map(category_order):
    """Map every possible loose/inconsistent spelling of a label in
    `category_order` back to its single canonical (standardized) spelling.
    """
    return {_normalize_label(label): label for label in category_order}


def compute_bid_category_pct(gdf, category_col, id_cols=ID_COLS, category_order=None):
    """Return a wide dataframe of per-BID percentage makeup of `category_col`.

    Rows with a null value in `category_col` are dropped before computing.
    Output columns: id_cols..., <category1>, <category2>, ..., pct_sum_check
    (pct_sum_check is a QA column, dropped before saving to CSV; each row should
    sum to ~100). Category columns are named after the raw category values —
    no prefix.

    If `category_order` is given, raw values are first standardized to match
    the exact spelling in `category_order` (via a whitespace/hyphen-tolerant
    match), and the resulting columns are ordered accordingly. Any raw value
    that doesn't match an entry in `category_order` is left as-is and appended
    (alphabetically) after the ordered columns, with a warning printed.
    """
    id_cols = list(id_cols)

    if category_col not in gdf.columns:
        raise KeyError(
            f"'{category_col}' not found. Available columns: {list(gdf.columns)}"
        )

    subset = gdf.dropna(subset=[category_col]).copy()
    n_dropped = len(gdf) - len(subset)
    print(f"[{category_col}] dropped {n_dropped} rows with null values "
          f"({len(subset)} remaining)")

    if category_order is not None:
        category_map = build_category_map(category_order)
        raw_values = subset[category_col]
        standardized = raw_values.apply(
            lambda v: category_map.get(_normalize_label(v), v)
        )
        unmatched = sorted(set(standardized) - set(category_order))
        if unmatched:
            print(f"[{category_col}] warning: {len(unmatched)} value(s) did not "
                  f"match `category_order` and were left as-is: {unmatched}")
        subset[category_col] = standardized

    counts = (
        subset
        .groupby(id_cols + [category_col])
        .size()
        .rename("count")
        .reset_index()
    )

    totals = counts.groupby(id_cols)["count"].sum().rename("total")
    counts = counts.merge(totals, on=id_cols)
    counts["pct"] = 100 * counts["count"] / counts["total"]

    wide = (
        counts
        .pivot(index=id_cols, columns=category_col, values="pct")
        .fillna(0)
    )
    wide.columns.name = None

    if category_order is not None:
        ordered_cols = [c for c in category_order if c in wide.columns]
        leftover_cols = sorted(c for c in wide.columns if c not in category_order)
        wide = wide[ordered_cols + leftover_cols]

    category_cols = list(wide.columns)
    wide = wide.reset_index()

    # QA check, not saved to the CSV
    wide["pct_sum_check"] = wide[category_cols].sum(axis=1)

    return wide


## 7. Run for every field and save one CSV each

Each output CSV keeps `FID`/`BID` as plain columns (no geometry), so it can be joined back onto the BID polygon file later using either key.

In [ ]:
EMPLOYEE_RANGE_ORDER = [
    "1 to 4",
    "5 to 9",
    "10 to 19",
    "20 to 49",
    "50 to 99",
    "100 to 249",
    "250 to 499",
    "500 to 999",
    "1000 to 4999",
]

# Reversed (smallest to largest); standardized spacing puts a single space on
# each side of every hyphen.
SALES_RANGE_ORDER = [
    "Less Than $500,000",
    "$500,000 - 1 Million",
    "$1 - 2.5 Million",
    "$2.5 - 5 Million",
    "$5 - 10 Million",
    "$10 - 20 Million",
    "$20 - 50 Million",
    "$50 - 100 Million",
    "$100 - 500 Million",
    "$500m - $1 Billion",
]

# Field -> explicit category order/standardization (None = leave as-is)
category_fields = {
    "broad_sector": None,
    "employee_range": EMPLOYEE_RANGE_ORDER,
    "sales_range": SALES_RANGE_ORDER,
    "own_or_lease_da": None,
}

category_results = {}

for field, order in category_fields.items():
    wide = compute_bid_category_pct(businesses_in_bids, field, category_order=order)
    category_results[field] = wide

    output_path = f"../../static/business_statistics/{field}_distribution.csv"
    wide.drop(columns=["pct_sum_check"]).to_csv(output_path, index=False)

    print(f"Saved {len(wide)} rows to {output_path}")
    print("Columns:", list(wide.drop(columns=['pct_sum_check']).columns))
    print()


In [ ]:
# Quick peek at each result
for field, wide in category_results.items():
    print(f"--- {field} ---")
    display(wide.head())